# 🌸 Jun OS — run it on a free Google Colab GPU

Jun OS is a chat app with an animated **Live2D character**, powered by a local AI model and (optionally) a real voice.

**How to use this notebook — 3 steps:**
0. Customize your preference for TTS/Model (Optional)
1. (If you see Connect T4 Skip this) Turn on the GPU: **Runtime → Change runtime type → T4 GPU → Save**.
2. Run the notebook (**Runtime → Run all**).
3. Scroll down to Step 3, click the link it prints. That's it. 🎉

Each step takes a few minutes the first time (it's downloading the AI model). Just wait for the ✅.

In [ ]:
#@title ▶️ Step 1 — Install everything  (~1 min)
#@markdown Sets up the web app and the AI engine. Just run it.
import os, subprocess, time, urllib.request

# Is a GPU available? (Jun runs without one, just slower.)
gpu = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip()
print("GPU:", gpu if gpu else "none — for best speed use Runtime > Change runtime type > GPU")

# Download the project
REPO_DIR = "/content/Jun"
if not os.path.isdir(REPO_DIR):
    !git clone -q --depth 1 https://github.com/efficiencyx/Jun.git {REPO_DIR}
%cd {REPO_DIR}

# PHP runs the web app + API; Ollama runs the AI model
print("Installing PHP and Ollama (this is the slow part)...")
!apt-get -qq update
!apt-get -qq install -y php-cli php-mbstring php-sqlite3 php-curl zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1

# Start the AI engine in the background
env = os.environ.copy()
env["OLLAMA_HOST"] = "127.0.0.1:11434"
subprocess.Popen(["ollama", "serve"], env=env,
                 stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=2)
        print("\n✅ Done. Go to Step 2."); break
    except Exception:
        time.sleep(1)
else:
    print("\n⚠️ The AI engine didn't start. See /content/ollama.log")

In [ ]:
#@title ▶️ Step 2 — Download the AI model  (a few GB, ~3–6 min)
Model = "auto"  #@param ["auto", "7B (smaller, faster)", "14B (bigger, smarter)"]
# @markdown `auto` picks 14B since Colab is capable of running it, you can otherwise use 7B for faster generation.
import subprocess

JUN_7B  = "hf.co/efficiencyx/Jun:Q4_K_M"
JUN_14B = "hf.co/efficiencyx/Jun-14B:Q4_K_M"

def vram_mb():
    try:
        out = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=memory.total", "--format=csv,noheader,nounits"])
        return int(out.decode().splitlines()[0])
    except Exception:
        return 0

if Model.startswith("7B"):
    MODEL = JUN_7B
elif Model.startswith("14B"):
    MODEL = JUN_14B
else:
    MODEL = JUN_14B if vram_mb() >= 12000 else JUN_7B

print(f"Downloading {MODEL} ...")
!ollama pull {MODEL}
!ollama pull nomic-embed-text          # lets Jun remember past chats & stay in character
!php tools/build_voice_index.php > /dev/null 2>&1 || true   # optional voice-style tuning
print("\n✅ Model ready. Go to Step 3.")

In [ ]:
#@title ▶️ Step 3 — Start Jun and get your link
Voice = True  #@param {type:"boolean"}
#@markdown Turn on text-to-speech so Jun talks out loud (adds ~2 min on first run).
import subprocess, os, time, re, urllib.request, urllib.error

PORT = 8000   # note: not 8080 — Colab reserves that port internally
DOCROOT = "/content/Jun/webapp"

# Optional voice engine (Kokoro) on :8001
if Voice:
    print("Setting up voice (downloads in the background)...")
    !apt-get -qq install -y espeak-ng > /dev/null
    !pip -q install -r /content/Jun/tts/requirements.txt
    subprocess.Popen(["python", "server.py"], cwd="/content/Jun/tts",
                     stdout=open("/content/kokoro.log", "w"), stderr=subprocess.STDOUT)

# Serve the web app with PHP
subprocess.run(["pkill", "-9", "-f", "php -S"], check=False)
try: os.remove(os.path.join(DOCROOT, "router.php"))   # leftover from older runs
except FileNotFoundError: pass
env = os.environ.copy()
env.update(OLLAMA_URL="http://127.0.0.1:11434",
           KOKORO_URL="http://127.0.0.1:8001",
           DEFAULT_MODEL=MODEL)
subprocess.Popen(["php", "-S", f"127.0.0.1:{PORT}", "-t", DOCROOT], env=env,
                 stdout=open("/content/php.log", "w"), stderr=subprocess.STDOUT)
time.sleep(2)

# Open a free public link with a Cloudflare tunnel (no signup)
subprocess.run(["pkill", "-9", "-f", "cloudflared"], check=False); time.sleep(1)
if not os.path.exists("/usr/local/bin/cloudflared"):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
    !chmod +x /usr/local/bin/cloudflared
subprocess.Popen(["cloudflared", "tunnel", "--no-autoupdate", "--url", f"http://127.0.0.1:{PORT}"],
                 stdout=open("/content/cloudflared.log", "w"), stderr=subprocess.STDOUT)

# Wait for the link, then wait until it actually answers (so it's not a dead link)
url = None
for _ in range(40):
    time.sleep(1)
    try:
        m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", open("/content/cloudflared.log").read())
        if m: url = m.group(0); break
    except FileNotFoundError:
        pass
if url:
    for _ in range(30):
        try:
            urllib.request.urlopen(url, timeout=5); break
        except urllib.error.HTTPError:
            break
        except Exception:
            time.sleep(2)

print("\n" + "="*60)
if url:
    print("  🌸 Open Jun OS here:\n")
    print("     " + url + "\n")
    print("  First visit: create an account (you must confirm you're an adult),")
    print("  then start chatting. The very first reply takes ~1–2 min while")
    print("  the model warms up; after that it's quick.")
else:
    print("  Couldn't create a link. Re-run this cell, or check")
    print("  /content/cloudflared.log")
print("="*60)

## If something goes wrong

- **The link doesn't open / says "unknown host":** wait ~30 seconds and reload, or just re-run Step 3 to get a fresh link.
- **First message is slow:** that's normal — the model is loading. Replies after that are fast.
- **Your account / chats disappear:** Colab wipes everything when the session ends, so accounts and history don't carry over between runs. That's expected for a free demo.
- **Need details?** Logs live in `/content/` (`ollama.log`, `php.log`, `kokoro.log`, `cloudflared.log`). View one with e.g. `!tail -n 40 /content/php.log`.